# H2分子激发态计算

本notebook使用`h2_excited_states.py`模块计算H2分子的基态、第一激发态和第二激发态。

In [1]:
import sys
sys.path.append('.')  # 添加当前目录到路径

from h2_excited_states import (
    setup_h2_system,
    compute_ground_state,
    compute_1st_excited_state,
    compute_2nd_excited_state,
    compute_all_excited_states,
    plot_convergence,
    plot_energy_levels
)

import matplotlib.pyplot as plt
import numpy as np

/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/netket/utils/dispatch.py:25: FutureWarning: 
The variables `nk.utils.dispatch.{TrueT|FalseT|Bool}` are deprecated. Their usages
should instead be replaced by the following objects:

    `TrueT` should be replaced by `typing.Literal[True]`
    `FalseT` should be replaced by `typing.Literal[False]`
    `Bool` should be replaced by `bool`

  _warn_deprecation(
/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/netket/driver/vmc_common.py:33: FutureWarning: 

            `nk.driver.vmc_common is deprecated and the functionality removed.   

If you imported `nk.driver.vmc_common`, you must reimplement that functionality yourself.


  warn_deprecation(


## 方法1：一键计算所有状态（推荐）

使用`compute_all_excited_states`函数一次性计算基态、第一激发态和第二激发态。

In [ ]:
results = compute_all_excited_states(
    bond_length=0.74,           # H2分子的键长（埃）
    n_iter=2000,                # 迭代次数
    alpha=2,                    # RBM的alpha参数
    learning_rate=0.03,          # 学习率
    diag_shift=0.01,            # SR的对角位移
    n_discard_per_chain=100,     # 每条链丢弃的样本数
    n_samples=1024,              # 样本数
    shift=2.3,                  # 正交化约束的位移参数（关键参数！）
    save_params=True,             # 保存参数到文件
    output_dir='Data'             # 输出目录
)

## 可视化结果

In [ ]:
# 绘制所有状态的收敛曲线
output_paths = [
    'Data/h2_ground_state',
    'Data/h2_excited_state_1',
    'Data/h2_excited_state_2'
]

plot_convergence(
    output_paths=output_paths,
    e_fci_all=results['e_fci_all'],
    E_fci=results['E_fci'],
    labels=['Ground State', '1st Excited State', '2nd Excited State']
)

In [ ]:
# 绘制能级图
energies = [
    results['final_energy_gs'],
    results['final_energy_ex1'],
    results['final_energy_ex2']
]

plot_energy_levels(
    energies=energies,
    e_fci_all=results['e_fci_all'],
    labels=['Ground State', '1st Excited', '2nd Excited']
)

## 方法2：分步计算

如果需要更精细的控制，可以分步计算每个状态。

In [4]:
# 步骤1：设置H2分子系统
mol, ha, hi, sampler, e_fci_all, E_fci = setup_h2_system(bond_length=0.74)

Hartree-Fock能量: -1.11675931 Ha
FCI基态能量: -1.13728383 Ha

FCI所有能级:
  E0 (基态) = -1.13728383 Ha
  E1 (第1激发态) = -0.53077336 Ha
  E2 (第2激发态) = -0.16835243 Ha
  E3 (第3激发态) = 0.48314267 Ha

Hilbert空间维度: 4
空间轨道数: 2
电子数: 2


In [5]:
# 步骤2：计算基态
vs_gs, final_energy_gs, gs_params = compute_ground_state(
    ha=ha,
    sampler=sampler,
    n_iter=500,
    alpha=2,
    learning_rate=0.1,
    diag_shift=0.01,
    n_discard_per_chain=50,
    n_samples=512,
    output_path='Data/h2_ground_state_step'
)


开始计算基态...


  0%|          | 0/500 [00:00<?, ?it/s]

100%|██████████| 500/500 [00:31<00:00, 15.89it/s, Energy=-1.137e+00-3.851e-10j ± 4.570e-27 [σ²=1.069e-50, R̂=0.9843]]    

基态能量: -1.13728384 Ha


In [6]:
# 步骤3：计算第一激发态
vs_ex1, final_energy_ex1, ex1_params, vs_gs = compute_1st_excited_state(
    ha=ha,
    sampler=sampler,
    gs_params=gs_params,
    e_fci=E_fci,
    n_iter=500,
    alpha=2,
    learning_rate=0.1,
    diag_shift=0.01,
    n_discard_per_chain=50,
    n_samples=512,
    shift=1.5,  # 关键参数：使用0.3而不是1.0
    output_path='Data/h2_excited_state_1_step'
)


开始计算第一激发态 (shift=1.5)...


100%|██████████| 500/500 [01:02<00:00,  8.01it/s, Energy=-0.53078101-0.00000335j ± 0.00000094 [σ²=0.00000001, R̂=1.0341]]

第一激发态能量: -0.53078101 Ha
激发能: 0.60650283 Ha


In [9]:
# 步骤4：计算第二激发态
vs_ex2, final_energy_ex2, ex2_params, vs_gs, vs_ex1_loaded = compute_2nd_excited_state(
    ha=ha,
    sampler=sampler,
    gs_params=gs_params,
    ex1_params=ex1_params,
    e_fci=E_fci,
    n_iter=500,
    alpha=2,
    learning_rate=0.1,
    diag_shift=0.01,
    n_discard_per_chain=50,
    n_samples=512,
    shift=1.5,  # 关键参数：使用0.3而不是1.0
    output_path='Data/h2_excited_state_2_step'
)


开始计算第二激发态 (shift=1.5)...


100%|██████████| 500/500 [01:42<00:00,  4.87it/s, Energy=-0.17199+0.00002j ± 0.00025 [σ²=0.00118, R̂=0.9763]]            

第二激发态能量: -0.17199077 Ha
激发能: 0.96529306 Ha


## 参数调优实验

测试不同的shift参数对结果的影响。

In [ ]:
# 测试不同的shift参数
shift_values = [0.1, 0.3, 0.5, 1.0]
results_shift = {}

for shift in shift_values:
    print(f"\n{'='*60}")
    print(f"测试 shift = {shift}")
    print('='*60)
    
    result = compute_all_excited_states(
        bond_length=1.4,
        n_iter=2000,
        alpha=2,
        learning_rate=0.03,
        diag_shift=0.01,
        n_discard_per_chain=100,
        n_samples=1024,
        shift=shift,
        save_params=False,  # 不保存参数以节省时间
        output_dir=f'Data/shift_{shift}'
    )
    
    results_shift[shift] = result

# 比较不同shift参数的结果
print("\n" + "="*60)
print("不同shift参数的结果比较")
print("="*60)
print(f"{'Shift':<10} {'E0':<15} {'E1':<15} {'E2':<15} {'Error E1':<15} {'Error E2':<15}")
print("-"*90)

for shift, result in results_shift.items():
    error_e1 = abs(result['final_energy_ex1'] - result['e_fci_all'][1])
    error_e2 = abs(result['final_energy_ex2'] - result['e_fci_all'][2])
    print(f"{shift:<10.2f} {result['final_energy_gs']:<15.8f} "
          f"{result['final_energy_ex1']:<15.8f} {result['final_energy_ex2']:<15.8f} "
          f"{error_e1:<15.8f} {error_e2:<15.8f}")

## 键长扫描

计算不同键长下的H2分子能级。

In [ ]:
# 定义键长范围
bond_lengths = np.linspace(0.5, 2.5, 9)  # 从0.5埃到2.5埃
results_bond = {}

for bond_length in bond_lengths:
    print(f"\n{'='*60}")
    print(f"计算键长 = {bond_length:.2f} 埃")
    print('='*60)
    
    result = compute_all_excited_states(
        bond_length=bond_length,
        n_iter=2000,
        alpha=2,
        learning_rate=0.03,
        diag_shift=0.01,
        n_discard_per_chain=100,
        n_samples=1024,
        shift=0.3,
        save_params=False,
        output_dir=f'Data/bond_{bond_length:.2f}'
    )
    
    results_bond[bond_length] = result

# 绘制键长-能量曲线
plt.figure(figsize=(12, 6))

# VMC结果
plt.plot(bond_lengths, [results_bond[bl]['final_energy_gs'] for bl in bond_lengths], 
         'o-', label='VMC E0', linewidth=2, markersize=8)
plt.plot(bond_lengths, [results_bond[bl]['final_energy_ex1'] for bl in bond_lengths], 
         's-', label='VMC E1', linewidth=2, markersize=8)
plt.plot(bond_lengths, [results_bond[bl]['final_energy_ex2'] for bl in bond_lengths], 
         '^-', label='VMC E2', linewidth=2, markersize=8)

# FCI结果（使用第一个键长的FCI结果作为参考）
ref_e_fci_all = results_bond[bond_lengths[0]]['e_fci_all']
plt.axhline(y=ref_e_fci_all[0], color='r', linestyle='--', alpha=0.5, label='FCI E0')
plt.axhline(y=ref_e_fci_all[1], color='g', linestyle='--', alpha=0.5, label='FCI E1')
plt.axhline(y=ref_e_fci_all[2], color='b', linestyle='--', alpha=0.5, label='FCI E2')

plt.xlabel('Bond Length (Å)', fontsize=12)
plt.ylabel('Energy (Ha)', fontsize=12)
plt.title('H2 Molecular Energy vs Bond Length', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 结果总结

打印最终的详细结果。

In [ ]:
print("="*80)
print("H2 分子激发态计算结果总结")
print("="*80)
print(f"\n计算参数:")
print(f"  键长: 1.4 Å")
print(f"  迭代次数: 2000")
print(f"  RBM alpha: 2")
print(f"  学习率: 0.03")
print(f"  SR对角位移: 0.01")
print(f"  样本数: 1024")
print(f"  Shift参数: 0.3")

print(f"\n计算结果:")
print(f"  基态能量 (E0): {results['final_energy_gs']:.8f} Ha")
print(f"  第一激发态能量 (E1): {results['final_energy_ex1']:.8f} Ha")
print(f"  第二激发态能量 (E2): {results['final_energy_ex2']:.8f} Ha")

print(f"\n激发能:")
print(f"  第一激发能 (E1-E0): {results['final_energy_ex1'] - results['final_energy_gs']:.8f} Ha")
print(f"  第二激发能 (E2-E0): {results['final_energy_ex2'] - results['final_energy_gs']:.8f} Ha")

print(f"\nFCI参考值:")
print(f"  FCI基态能量: {results['E_fci']:.8f} Ha")
print(f"  FCI第一激发态能量: {results['e_fci_all'][1]:.8f} Ha")
print(f"  FCI第二激发态能量: {results['e_fci_all'][2]:.8f} Ha")

print(f"\n误差分析:")
print(f"  基态误差: {abs(results['final_energy_gs'] - results['E_fci']):.8f} Ha")
print(f"  第一激发态误差: {abs(results['final_energy_ex1'] - results['e_fci_all'][1]):.8f} Ha")
print(f"  第二激发态误差: {abs(results['final_energy_ex2'] - results['e_fci_all'][2]):.8f} Ha")

print(f"\n相对误差:")
print(f"  基态相对误差: {abs(results['final_energy_gs'] - results['E_fci'])/abs(results['E_fci'])*100:.4f}%")
print(f"  第一激发态相对误差: {abs(results['final_energy_ex1'] - results['e_fci_all'][1])/abs(results['e_fci_all'][1])*100:.4f}%")
print(f"  第二激发态相对误差: {abs(results['final_energy_ex2'] - results['e_fci_all'][2])/abs(results['e_fci_all'][2])*100:.4f}%")

print("="*80)